Data Transormation part-1

Topics to be covered:
1) Reading CSV Data
2) Understanding DataFrames
3) Common Transformations
4) Data Cleaning
5) Aggregations
6) Window Functions
7) Joins
8) Writing Transformed Data

In [106]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [107]:
# Create SparkSession:

spark = SparkSession.builder.appName("CSV Transformation").getOrCreate()

In [108]:
# Read CSV:
df = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
    )


In [109]:
# View Data:
df.show()

+---+-----+----------+------+----+
| id| name|department|salary| age|
+---+-----+----------+------+----+
|  1| NULL|        IT| 60000|NULL|
|  2|Alice|      NULL| 50000|  28|
|  3|  Bob|        IT| 70000|  35|
|  4| Emma|   Finance| 80000|  40|
|  5| NULL|     Sales| 55000|  29|
|  6|David|      NULL| 75000|  32|
|  3|  Bob|        IT| 70000|  35|
+---+-----+----------+------+----+



In [110]:
# Check Schema:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [111]:
# Selecting Specific Columns: 
df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
| NULL| 60000|
|Alice| 50000|
|  Bob| 70000|
| Emma| 80000|
| NULL| 55000|
|David| 75000|
|  Bob| 70000|
+-----+------+



In [112]:
# Other way to Select and show specific columns:
df.select(
    col("name"),
    col("salary")
).show()

+-----+------+
| name|salary|
+-----+------+
| NULL| 60000|
|Alice| 50000|
|  Bob| 70000|
| Emma| 80000|
| NULL| 55000|
|David| 75000|
|  Bob| 70000|
+-----+------+



In [113]:
# Rename Columns:
df.select(
    col("salary").alias("annual_salary")
).show()

+-------------+
|annual_salary|
+-------------+
|        60000|
|        50000|
|        70000|
|        80000|
|        55000|
|        75000|
|        70000|
+-------------+



In [114]:
# Filtering Data:

# Employees with salary > 60000:

df.filter(df.salary > 60000).show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  3|  Bob|        IT| 70000| 35|
|  4| Emma|   Finance| 80000| 40|
|  6|David|      NULL| 75000| 32|
|  3|  Bob|        IT| 70000| 35|
+---+-----+----------+------+---+



In [115]:
# Employees with salary > 60000:
df.where("salary > 60000").show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  3|  Bob|        IT| 70000| 35|
|  4| Emma|   Finance| 80000| 40|
|  6|David|      NULL| 75000| 32|
|  3|  Bob|        IT| 70000| 35|
+---+-----+----------+------+---+



In [116]:
# Multiple Conditions:
# Get the Employees whose salary is above 50000 and dept is IT.

df.filter(
    (df.salary > 50000) &
    (df.department == "IT")
).show()

+---+----+----------+------+----+
| id|name|department|salary| age|
+---+----+----------+------+----+
|  1|NULL|        IT| 60000|NULL|
|  3| Bob|        IT| 70000|  35|
|  3| Bob|        IT| 70000|  35|
+---+----+----------+------+----+



In [117]:
# Adding new column: 

# Increase the salary of employees by 10% :

df = df.withColumn(
    "new_salary",
    col("salary") * 1.10
)

df.show()

+---+-----+----------+------+----+-----------------+
| id| name|department|salary| age|       new_salary|
+---+-----+----------+------+----+-----------------+
|  1| NULL|        IT| 60000|NULL|          66000.0|
|  2|Alice|      NULL| 50000|  28|55000.00000000001|
|  3|  Bob|        IT| 70000|  35|          77000.0|
|  4| Emma|   Finance| 80000|  40|          88000.0|
|  5| NULL|     Sales| 55000|  29|60500.00000000001|
|  6|David|      NULL| 75000|  32|          82500.0|
|  3|  Bob|        IT| 70000|  35|          77000.0|
+---+-----+----------+------+----+-----------------+



In [118]:
# String Transformation:

df.withColumn(
    "name_upper",
    upper(col("name"))
).show()

+---+-----+----------+------+----+-----------------+----------+
| id| name|department|salary| age|       new_salary|name_upper|
+---+-----+----------+------+----+-----------------+----------+
|  1| NULL|        IT| 60000|NULL|          66000.0|      NULL|
|  2|Alice|      NULL| 50000|  28|55000.00000000001|     ALICE|
|  3|  Bob|        IT| 70000|  35|          77000.0|       BOB|
|  4| Emma|   Finance| 80000|  40|          88000.0|      EMMA|
|  5| NULL|     Sales| 55000|  29|60500.00000000001|      NULL|
|  6|David|      NULL| 75000|  32|          82500.0|     DAVID|
|  3|  Bob|        IT| 70000|  35|          77000.0|       BOB|
+---+-----+----------+------+----+-----------------+----------+



In [119]:
df.withColumn(
    "name_lower",
    lower(col("name"))
).show()

+---+-----+----------+------+----+-----------------+----------+
| id| name|department|salary| age|       new_salary|name_lower|
+---+-----+----------+------+----+-----------------+----------+
|  1| NULL|        IT| 60000|NULL|          66000.0|      NULL|
|  2|Alice|      NULL| 50000|  28|55000.00000000001|     alice|
|  3|  Bob|        IT| 70000|  35|          77000.0|       bob|
|  4| Emma|   Finance| 80000|  40|          88000.0|      emma|
|  5| NULL|     Sales| 55000|  29|60500.00000000001|      NULL|
|  6|David|      NULL| 75000|  32|          82500.0|     david|
|  3|  Bob|        IT| 70000|  35|          77000.0|       bob|
+---+-----+----------+------+----+-----------------+----------+



In [120]:
# Trim Spaces:
df.withColumn(
    "name_trimmed",
    trim(col("name"))
).show()

+---+-----+----------+------+----+-----------------+------------+
| id| name|department|salary| age|       new_salary|name_trimmed|
+---+-----+----------+------+----+-----------------+------------+
|  1| NULL|        IT| 60000|NULL|          66000.0|        NULL|
|  2|Alice|      NULL| 50000|  28|55000.00000000001|       Alice|
|  3|  Bob|        IT| 70000|  35|          77000.0|         Bob|
|  4| Emma|   Finance| 80000|  40|          88000.0|        Emma|
|  5| NULL|     Sales| 55000|  29|60500.00000000001|        NULL|
|  6|David|      NULL| 75000|  32|          82500.0|       David|
|  3|  Bob|        IT| 70000|  35|          77000.0|         Bob|
+---+-----+----------+------+----+-----------------+------------+



In [121]:
df.withColumn(
    "emp_info",
    concat(col("name"), lit("-"), col("department"))
).show()

+---+-----+----------+------+----+-----------------+------------+
| id| name|department|salary| age|       new_salary|    emp_info|
+---+-----+----------+------+----+-----------------+------------+
|  1| NULL|        IT| 60000|NULL|          66000.0|        NULL|
|  2|Alice|      NULL| 50000|  28|55000.00000000001|        NULL|
|  3|  Bob|        IT| 70000|  35|          77000.0|      Bob-IT|
|  4| Emma|   Finance| 80000|  40|          88000.0|Emma-Finance|
|  5| NULL|     Sales| 55000|  29|60500.00000000001|        NULL|
|  6|David|      NULL| 75000|  32|          82500.0|        NULL|
|  3|  Bob|        IT| 70000|  35|          77000.0|      Bob-IT|
+---+-----+----------+------+----+-----------------+------------+



In [122]:
spark.stop()

In [123]:
spark = SparkSession.builder.appName("csv_transformation2").getOrCreate()

df1 = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
)

df1.show()

+---+-----+----------+------+----+
| id| name|department|salary| age|
+---+-----+----------+------+----+
|  1| NULL|        IT| 60000|NULL|
|  2|Alice|      NULL| 50000|  28|
|  3|  Bob|        IT| 70000|  35|
|  4| Emma|   Finance| 80000|  40|
|  5| NULL|     Sales| 55000|  29|
|  6|David|      NULL| 75000|  32|
|  3|  Bob|        IT| 70000|  35|
+---+-----+----------+------+----+



In [124]:
# Check Nulls:

df1.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df1.columns
]).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  0|   2|         2|     0|  1|
+---+----+----------+------+---+



In [125]:
# Drop null rows:
df1.na.drop().show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  3| Bob|        IT| 70000| 35|
|  4|Emma|   Finance| 80000| 40|
|  3| Bob|        IT| 70000| 35|
+---+----+----------+------+---+



In [126]:
df1.na.fill({
    "name": "Unkown",
    "department": "Unkown",
    "age": 0
}).show()

+---+------+----------+------+---+
| id|  name|department|salary|age|
+---+------+----------+------+---+
|  1|Unkown|        IT| 60000|  0|
|  2| Alice|    Unkown| 50000| 28|
|  3|   Bob|        IT| 70000| 35|
|  4|  Emma|   Finance| 80000| 40|
|  5|Unkown|     Sales| 55000| 29|
|  6| David|    Unkown| 75000| 32|
|  3|   Bob|        IT| 70000| 35|
+---+------+----------+------+---+



In [127]:
df1.show()

+---+-----+----------+------+----+
| id| name|department|salary| age|
+---+-----+----------+------+----+
|  1| NULL|        IT| 60000|NULL|
|  2|Alice|      NULL| 50000|  28|
|  3|  Bob|        IT| 70000|  35|
|  4| Emma|   Finance| 80000|  40|
|  5| NULL|     Sales| 55000|  29|
|  6|David|      NULL| 75000|  32|
|  3|  Bob|        IT| 70000|  35|
+---+-----+----------+------+----+



In [128]:
# Remove Duplicates:

# df1.dropDuplicates(["id"]).show()
# df1.dropDuplicates(["department"]).show()


In [129]:
# Type Conversion:

# Convert salary to double
df2 = df1.withColumn(
    "salary",
    col("salary").cast(DoubleType())
)

df1.printSchema()
df2.printSchema()
df2.show()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- age: integer (nullable = true)

+---+-----+----------+-------+----+
| id| name|department| salary| age|
+---+-----+----------+-------+----+
|  1| NULL|        IT|60000.0|NULL|
|  2|Alice|      NULL|50000.0|  28|
|  3|  Bob|        IT|70000.0|  35|
|  4| Emma|   Finance|80000.0|  40|
|  5| NULL|     Sales|55000.0|  29|
|  6|David|      NULL|75000.0|  32|
|  3|  Bob|        IT|70000.0|  35|
+---+-----+----------+-------+----+



In [130]:
# Sorting Data 
df2.orderBy("salary").show()
df2.orderBy("age").show()

+---+-----+----------+-------+----+
| id| name|department| salary| age|
+---+-----+----------+-------+----+
|  2|Alice|      NULL|50000.0|  28|
|  5| NULL|     Sales|55000.0|  29|
|  1| NULL|        IT|60000.0|NULL|
|  3|  Bob|        IT|70000.0|  35|
|  3|  Bob|        IT|70000.0|  35|
|  6|David|      NULL|75000.0|  32|
|  4| Emma|   Finance|80000.0|  40|
+---+-----+----------+-------+----+

+---+-----+----------+-------+----+
| id| name|department| salary| age|
+---+-----+----------+-------+----+
|  1| NULL|        IT|60000.0|NULL|
|  2|Alice|      NULL|50000.0|  28|
|  5| NULL|     Sales|55000.0|  29|
|  6|David|      NULL|75000.0|  32|
|  3|  Bob|        IT|70000.0|  35|
|  3|  Bob|        IT|70000.0|  35|
|  4| Emma|   Finance|80000.0|  40|
+---+-----+----------+-------+----+



In [131]:
# Order by multiple columns 

df2.orderBy(
    col("department"),
    col("salary").desc()
).show()

+---+-----+----------+-------+----+
| id| name|department| salary| age|
+---+-----+----------+-------+----+
|  6|David|      NULL|75000.0|  32|
|  2|Alice|      NULL|50000.0|  28|
|  4| Emma|   Finance|80000.0|  40|
|  3|  Bob|        IT|70000.0|  35|
|  3|  Bob|        IT|70000.0|  35|
|  1| NULL|        IT|60000.0|NULL|
|  5| NULL|     Sales|55000.0|  29|
+---+-----+----------+-------+----+



In [132]:
# Group By and Aggregation:

# Departmentwise average Salary:

df2.groupBy("department").avg("salary").show()

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|     Sales|          55000.0|
|      NULL|          62500.0|
|   Finance|          80000.0|
|        IT|66666.66666666667|
+----------+-----------------+



In [133]:
# Multiple Aggregations:

df2.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary"),
    count("*").alias("employee_count")
).show()

+----------+-----------------+----------+----------+--------------+
|department|       avg_salary|max_salary|min_salary|employee_count|
+----------+-----------------+----------+----------+--------------+
|     Sales|          55000.0|   55000.0|   55000.0|             1|
|      NULL|          62500.0|   75000.0|   50000.0|             2|
|   Finance|          80000.0|   80000.0|   80000.0|             1|
|        IT|66666.66666666667|   70000.0|   60000.0|             3|
+----------+-----------------+----------+----------+--------------+



In [137]:
# Window Functions:

window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

# window_spec

df2.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+---+-----+----------+-------+----+----+
| id| name|department| salary| age|rank|
+---+-----+----------+-------+----+----+
|  6|David|      NULL|75000.0|  32|   1|
|  2|Alice|      NULL|50000.0|  28|   2|
|  4| Emma|   Finance|80000.0|  40|   1|
|  3|  Bob|        IT|70000.0|  35|   1|
|  3|  Bob|        IT|70000.0|  35|   1|
|  1| NULL|        IT|60000.0|NULL|   3|
|  5| NULL|     Sales|55000.0|  29|   1|
+---+-----+----------+-------+----+----+



In [145]:
# Join CSV Data:
emp_df = spark.read.csv(
    "dataFiles/emp.csv",
    header=True,
    inferSchema=True
)


dept_df = spark.read.csv(
    "dataFiles/dept.csv",
    header=True,
    inferSchema=True
)

emp_df.show()
dept_df.show()

+---+-----+
| id| name|
+---+-----+
|  1| John|
|  2|Alice|
|  3|David|
+---+-----+

+---+----------+
| id|department|
+---+----------+
|  1|        IT|
|  2|        HR|
|  4|     Sales|
+---+----------+



In [150]:
# Inner Join:

result_inner = emp_df.join(
    dept_df,
    "id",
    "inner"
)

result_inner.show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1| John|        IT|
|  2|Alice|        HR|
+---+-----+----------+



In [147]:
# Left Join:

result_left = emp_df.join(
    dept_df,
    "id",
    "left"
)

result_left.show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1| John|        IT|
|  2|Alice|        HR|
|  3|David|      NULL|
+---+-----+----------+



In [148]:
result_right = emp_df.join(
    dept_df,
    "id",
    "right"
)

result_right.show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1| John|        IT|
|  2|Alice|        HR|
|  4| NULL|     Sales|
+---+-----+----------+



In [149]:
result_full = emp_df.join(
    dept_df,
    "id",
    "full"
)

result_full.show()

+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1| John|        IT|
|  2|Alice|        HR|
|  3|David|      NULL|
|  4| NULL|     Sales|
+---+-----+----------+



In [ ]:
# Using SQL on CSV Data:
# Create temperory Views

df2.createOrReplaceTempView(
    "employ"
)

res = spark.sql("""
    SELECT
        department,
        AVG(salary) avg_salary
        FROM employ
        Group By department
""")

res.show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|     Sales|          55000.0|
|      NULL|          62500.0|
|   Finance|          80000.0|
|        IT|66666.66666666667|
+----------+-----------------+



In [155]:
res1 = spark.sql("""
        select
                 *
        from
                 employ
""")

res1.show()

+---+-----+----------+-------+----+
| id| name|department| salary| age|
+---+-----+----------+-------+----+
|  1| NULL|        IT|60000.0|NULL|
|  2|Alice|      NULL|50000.0|  28|
|  3|  Bob|        IT|70000.0|  35|
|  4| Emma|   Finance|80000.0|  40|
|  5| NULL|     Sales|55000.0|  29|
|  6|David|      NULL|75000.0|  32|
|  3|  Bob|        IT|70000.0|  35|
+---+-----+----------+-------+----+

